In [ ]:
import os
import ast
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import proplot as pplt
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi': 900,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
    'tick.minor': False,
    'font.size': 9,
    'label.size': 9,
    'tick.labelsize': 9,
    'title.size': 9,
    'abc.size': 9,
    'legend.fontsize': 9,
    'suptitle.size': 9,
    'leftlabelsize': 9,
    'toplabelsize': 9,
    'leftlabel.weight': 'normal',
    'toplabel.weight': 'normal',
    'reso': 'xx-hi'})

In [ ]:
with open('../scripts/configs.json', 'r', encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR    = CONFIGS['filepaths']['splits']
WEIGHTSDIR   = CONFIGS['filepaths']['weights']
MODELSDIR    = CONFIGS['filepaths']['models']
PREDSDIR     = CONFIGS['filepaths']['predictions']
RAWDIR       = CONFIGS['filepaths']['raw']
MODELS       = CONFIGS['experiments']
LATRANGE     = CONFIGS['domain']['latrange']
LONRANGE     = CONFIGS['domain']['lonrange']
FIELDVARS    = MODELS['sr']['runs']['sr_gauss']['fieldvars']
NNSEEDS      = MODELS['nn']['seeds']
SRSEEDS      = MODELS['sr']['seeds']
OPTIMIZEDEQS = MODELS['sr']['optimizedeqs']
SPLIT        = 'test'
NBINS        = 20
MINSAMPLES   = 50
ORDER        = ['sr_med', 'sr_hi', 'nn_gauss']
SRFUNCTIONS  = {
    'cube': lambda x: x**3, 'square': lambda x: x**2, 'neg': lambda x: -x,
    'sqrt': np.sqrt, 'exp': np.exp, 'log': np.log, 'abs': np.abs,
    'max': np.maximum, 'min': np.minimum}
COLORS = {}
LABELS = {}
for name, config in {**MODELS['nn']['runs'], **OPTIMIZEDEQS}.items():
    COLORS[name] = config['color']
    LABELS[name] = config['description']
KWMAP = dict(coast=True, latlim=LATRANGE, lonlim=LONRANGE,
             latlines=[10, 15, 20], lonlines=[65, 75, 85], grid=False)

In [ ]:
def kernel_integrate(fields, weights, dsig, mask=None):
    w = fields * weights[None, :, :] * dsig[None, None, :]
    if mask is not None:
        w = w * mask[:, None, :]
    return w.sum(axis=2)

def to_phys(norm):
    return np.expm1(tpstd * np.maximum(0.0, np.asarray(norm, dtype=float)))

def eval_form(form, variables, constants):
    ns = dict(SRFUNCTIONS)
    ns.update(variables)
    ns.update(constants)
    return np.asarray(eval(form, {'__builtins__': {}}, ns), dtype=float)

def used_predictors(form, candidates):
    names = {n.id for n in ast.walk(ast.parse(form, mode='eval')) if isinstance(n, ast.Name)}
    return [c for c in candidates if c in names]

def r2(ytrue, ypred):
    mask = np.isfinite(ytrue) & np.isfinite(ypred)
    o, p = ytrue[mask], ypred[mask]
    return 1 - np.sum((o - p)**2) / np.sum((o - o.mean())**2)

def bin_1d(x, z, nbins=NBINS, minsamples=MINSAMPLES, plo=1, phi=99):
    finite = np.isfinite(x) & np.isfinite(z)
    x, z = x[finite], z[finite]
    edges = np.linspace(*np.percentile(x, [plo, phi]), nbins + 1)
    n = len(edges) - 1
    xi = np.clip(np.digitize(x, edges) - 1, 0, n - 1)
    counts = np.bincount(xi, minlength=n)
    sums = np.bincount(xi, weights=z, minlength=n)
    return 0.5 * (edges[:-1] + edges[1:]), np.where(counts >= minsamples, sums / counts, np.nan), counts

def bin_2d(x, y, z, nbins=NBINS, minsamples=MINSAMPLES, plo=1, phi=99):
    finite = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    x, y, z = x[finite], y[finite], z[finite]
    xedges = np.linspace(*np.percentile(x, [plo, phi]), nbins + 1)
    yedges = np.linspace(*np.percentile(y, [plo, phi]), nbins + 1)
    xi = np.clip(np.digitize(x, xedges) - 1, 0, nbins - 1)
    yi = np.clip(np.digitize(y, yedges) - 1, 0, nbins - 1)
    idx = xi * nbins + yi
    counts = np.bincount(idx, minlength=nbins * nbins).reshape(nbins, nbins)
    sums = np.bincount(idx, weights=z, minlength=nbins * nbins).reshape(nbins, nbins)
    return (0.5 * (xedges[:-1] + xedges[1:]), 0.5 * (yedges[:-1] + yedges[1:]),
            np.where(counts >= minsamples, sums / counts, np.nan), counts)

def load_pred(name, split=SPLIT):
    path = os.path.join(PREDSDIR, f'{name}_{split}_predictions.nc')
    if not os.path.exists(path):
        return None
    with xr.open_dataset(path, engine='h5netcdf') as ds:
        da = ds['tp'].load()
    if 'seed' in da.dims:
        da = da.mean('seed')
    if 'complexity' in da.dims:
        da = da.isel(complexity=0)
    pred = da.transpose('time', 'lat', 'lon').values.ravel()
    return pred if pred.shape[0] == obsflat.shape[0] else None

def load_pareto(run, seeds=SRSEEDS):
    frames = []
    for seed in seeds:
        path = os.path.join(MODELSDIR, 'sr', f'{run}_{seed}_equations.csv')
        if os.path.exists(path):
            df = pd.read_csv(path)
            df['seed'] = seed
            frames.append(df)
    if not frames:
        return None
    df = pd.concat(frames, ignore_index=True)
    return df.groupby('complexity').agg(
        loss=('loss', 'mean'), equation=('equation', 'first')).reset_index()

def to_map(flat):
    return np.nanmean(flat.reshape(ntime, nlat, nlon), axis=0)

In [ ]:
# --- stats ---
with open(os.path.join(SPLITSDIR, 'stats.json'), 'r', encoding='utf-8') as f:
    stats = json.load(f)
tpmean = float(stats['tp_mean'])
tpstd  = float(stats['tp_std'])

def flatten(da):
    if 'time' in da.dims:
        return da.transpose('time', 'lat', 'lon').values.ravel()
    return np.tile(da.values, (ntime, 1, 1)).ravel()

# --- normalized split ---
with xr.open_dataset(os.path.join(SPLITSDIR, f'norm_{SPLIT}.h5'), engine='h5netcdf') as ds:
    ntime, nlat, nlon = ds.sizes['time'], ds.sizes['lat'], ds.sizes['lon']
    nsig = ds.sizes.get('sig', 1)
    lat  = ds['lat'].values
    lon  = ds['lon'].values
    dsig = ds['dsig'].values
    sig  = ds['sig'].values if 'sig' in ds else np.linspace(0.5, 1.0, nsig)
    fields   = np.stack([ds[v].transpose('time', 'lat', 'lon', 'sig').values.reshape(-1, nsig)
                         for v in FIELDVARS], axis=1)
    surfmask = (ds['surfmask'].transpose('time', 'lat', 'lon', 'sig').values.reshape(-1, nsig)
                if 'surfmask' in ds else None)
    blnorm  = flatten(ds['bl'])
    lfnorm  = flatten(ds['lf'])
    shfnorm = flatten(ds['shf'])
    lhfnorm = flatten(ds['lhf'])

# --- kernel integration ---
kernels = []
for seed in NNSEEDS:
    wpath = os.path.join(WEIGHTSDIR, f'nn_gauss_{seed}_weights.nc')
    if os.path.exists(wpath):
        with xr.open_dataset(wpath, engine='h5netcdf') as ds:
            kernels.append(ds['k'].values)
kmean = np.mean(kernels, axis=0) if kernels else None
ki = (np.mean([kernel_integrate(fields, k, dsig, surfmask) for k in kernels], axis=0)
      if kernels else fields.mean(axis=2))
rhk, thetaek, thetaestark = ki[:, 0], ki[:, 1], ki[:, 2]

# --- raw split ---
with xr.open_dataset(os.path.join(SPLITSDIR, f'{SPLIT}.h5'), engine='h5netcdf') as ds:
    obsflat = ds.tp.transpose('time', 'lat', 'lon').values.ravel()
    lfraw   = flatten(ds['lf'])
    shfraw  = flatten(ds['shf'])
    lhfraw  = flatten(ds['lhf'])
    sdoraw  = flatten(ds['sdo']) if 'sdo' in ds else None
    sefraw  = flatten(ds['sef']) if 'sef' in ds else None
    seraw   = flatten(ds['se'])  if 'se'  in ds else None
    months  = np.tile(ds.time.dt.month.values[:, None, None]
                      * np.ones((1, nlat, nlon)), 1).ravel().astype(int)

# --- orography ---
semap = None
sepath = os.path.join(RAWDIR, 'ERA5_surface_elevation.nc')
if os.path.exists(sepath):
    with xr.open_dataset(sepath, engine='netcdf4') as ds:
        sevar = list(ds.data_vars)[0]
        se = ds[sevar]
        if 'time' in se.dims:
            se = se.isel(time=0)
        se    = se.interp(lat=lat, lon=lon, method='linear')
        semap = se.values.astype(np.float32)
seflat = np.tile(semap, (ntime, 1, 1)).ravel() if semap is not None else seraw

# --- SR registry ---
regpath = os.path.join(MODELSDIR, 'sr', 'optimized_equations.pkl')
if os.path.exists(regpath):
    with open(regpath, 'rb') as f:
        SR_REGISTRY = pickle.load(f)
else:
    SR_REGISTRY = {}

# --- model predictions ---
VARS = {'rh': rhk, 'thetae': thetaek, 'thetaestar': thetaestark,
        'bl': blnorm, 'lf': lfnorm, 'shf': shfnorm, 'lhf': lhfnorm}

MODELPRED = {}
for name, config in OPTIMIZEDEQS.items():
    entry     = SR_REGISTRY.get(name, {})
    form      = entry.get('form', config['form'])
    constants = entry.get('constants', config['init'])
    pnames    = used_predictors(form, VARS.keys())
    if all(p in VARS for p in pnames):
        MODELPRED[name] = to_phys(eval_form(form, {p: VARS[p] for p in pnames}, constants))

for name in MODELS['nn']['runs']:
    pred = load_pred(name)
    if pred is not None:
        MODELPRED[name] = pred

valid = np.isfinite(obsflat)
for arr in VARS.values():
    valid &= np.isfinite(arr)

# --- feature data for plotting ---
VARLABELS = {
    'rh': r'KI-$\widehat{\mathrm{RH}}$',
    'thetae': r'KI-$\widehat{\theta_e}$',
    'thetaestar': r'KI-$\widehat{\theta_e^*}$',
    'bl': 'BL (norm)',
    'lf': 'Land Fraction',
    'shf': r'SHF (W m$^{-2}$)',
    'lhf': r'LHF (W m$^{-2}$)',
    'sdo': 'SDO (m)',
    'sef': 'SEF (m)',
    'se': 'SE (m)'}

VARDATA = {'rh': rhk, 'thetae': thetaek, 'thetaestar': thetaestark,
           'bl': blnorm, 'lf': lfraw, 'shf': shfraw, 'lhf': lhfraw}
if sdoraw is not None:
    VARDATA['sdo'] = sdoraw
if sefraw is not None:
    VARDATA['sef'] = sefraw
if seraw is not None:
    VARDATA['se'] = seraw

print(f'Valid: {valid.sum():,} | Loaded: {sorted(MODELPRED.keys())}')
for name in ORDER:
    if name in MODELPRED:
        print(f'  {LABELS.get(name, name):20s}  R²={r2(obsflat[valid], MODELPRED[name][valid]):.3f}')
    else:
        print(f'  {LABELS.get(name, name):20s}  — not available')

In [ ]:
# Fig 1 — Skill overview: overall R², by month, and by region
obs   = obsflat[valid]
names = [n for n in ORDER if n in MODELPRED]
r2vals = {LABELS[n]: r2(obs, MODELPRED[n][valid]) for n in names}

mons = months[valid]
latv = np.tile(lat[None, :, None], (ntime, 1, nlon)).ravel()[valid]
lonv = np.tile(lon[None, None, :], (ntime, nlat, 1)).ravel()[valid]
lfv  = lfraw[valid]
regions = {
    'Ocean': lfv < 0.1, 'Land': lfv > 0.9,
    '5–15°N': (latv >= 5) & (latv < 15),
    '15–25°N': (latv >= 15) & (latv <= 25),
    '60–75°E': (lonv >= 60) & (lonv < 75),
    '75–90°E': (lonv >= 75) & (lonv <= 90)}

fig, axs = pplt.subplots(nrows=1, ncols=3, refwidth=2.8, refheight=2, share=False)
clrs = [COLORS[n] for n in names]

axs[0].bar(np.arange(len(r2vals)), list(r2vals.values()), color=clrs, alpha=0.85)
axs[0].format(ylabel=r'$R^2$', xticks=np.arange(len(r2vals)),
              xticklabels=list(r2vals.keys()), xrotation=15,
              title=r'Overall $R^2$', grid=False, ylim=(0, 0.5))

xmon = np.array([6, 7, 8])
for name in names:
    pred = MODELPRED[name][valid]
    axs[1].plot(xmon, [r2(obs[mons == m], pred[mons == m]) for m in xmon],
                color=COLORS[name], linewidth=1.5, marker='o', markersize=4)
axs[1].format(xlabel='Month', ylabel=r'$R^2$', xticks=xmon,
              xticklabels=['Jun', 'Jul', 'Aug'], title=r'$R^2$ by Month', grid=False)

xreg  = np.arange(len(regions))
width = 0.8 / max(len(names), 1)
for i, name in enumerate(names):
    pred = MODELPRED[name][valid]
    axs[2].bar(xreg + i * width,
               [r2(obs[mask], pred[mask]) for mask in regions.values()],
               width=width, color=COLORS[name], alpha=0.85)
axs[2].format(ylabel=r'$R^2$', xticks=xreg + width,
              xticklabels=list(regions.keys()), xrotation=20,
              title=r'$R^2$ by Region', grid=False)

fig.format(suptitle='Skill Overview')
pplt.show()

In [ ]:
# Fig 2 — Spatial bias (top) and residual variance (bottom) per model
obsmap = to_map(obsflat)
obs3d  = obsflat.reshape(ntime, nlat, nlon)
names  = [n for n in ORDER if n in MODELPRED]
ncols  = 1 + len(names)

fig, axs = pplt.subplots(nrows=2, ncols=ncols, proj='cyl', refwidth=2, share=False)

mc = axs[0, 0].pcolormesh(lon, lat, obsmap, cmap='Blues', vmin=0, vmax=2, extend='max')
axs[0, 0].format(title='Observed', latlabels='l', lonlabels=False, **KWMAP)

mb = None
for i, name in enumerate(names):
    bmap = to_map(MODELPRED[name]) - obsmap
    mb = axs[0, i + 1].pcolormesh(lon, lat, bmap, cmap='DryWet',
                                   vmin=-0.5, vmax=0.5, extend='both')
    axs[0, i + 1].format(title=LABELS.get(name, name),
                          latlabels=False, lonlabels=False, **KWMAP)

var_maps = []
vmax_var = 0
for name in names:
    pred3d = MODELPRED[name].reshape(ntime, nlat, nlon)
    bmean  = np.nanmean(pred3d - obs3d, axis=0)
    vmap   = np.nanmean((pred3d - obs3d)**2, axis=0) - bmean**2
    var_maps.append(vmap)
    vmax_var = max(vmax_var, float(np.nanpercentile(vmap, 98)))

obs_std = np.nanstd(obs3d, axis=0)
axs[1, 0].pcolormesh(lon, lat, obs_std, cmap='Purples',
                      vmin=0, vmax=vmax_var, extend='max')
axs[1, 0].format(title=r'Obs $\sigma$', latlabels='l', lonlabels='b', **KWMAP)

mv = None
for i, (name, vmap) in enumerate(zip(names, var_maps)):
    mv = axs[1, i + 1].pcolormesh(lon, lat, vmap, cmap='Purples',
                                    vmin=0, vmax=vmax_var, extend='max')
    axs[1, i + 1].format(title=LABELS.get(name, name),
                          latlabels=False, lonlabels='b', **KWMAP)

fig.colorbar(mc, loc='b', col=1, label='Precipitation (mm)')
if mb:
    fig.colorbar(mb, loc='b', col=(2, ncols), label='Bias: Pred − Obs (mm)')
if mv:
    fig.colorbar(mv, loc='r', label=r'Variance (mm$^2$)')
fig.format(suptitle='Spatial Bias and Variance',
           rowlabels=['Bias', 'Variance'])
pplt.show()

In [ ]:
# Fig 3 — SDO and surface elevation maps
sdomap = sdoraw.reshape(ntime, nlat, nlon)[0] if sdoraw is not None else None

panels = []
if sdomap is not None:
    panels.append(('SDO (m)', sdomap, 'YlOrBr'))
if semap is not None:
    panels.append(('Surface Elevation (m)', semap, 'Greens'))

if panels:
    fig, axs = pplt.subplots(nrows=1, ncols=len(panels), proj='cyl',
                              refwidth=3, share=False)
    axsf = np.atleast_1d(axs).ravel()
    for i, (ax, (title, data, cmap)) in enumerate(zip(axsf, panels)):
        m = ax.pcolormesh(lon, lat, data, cmap=cmap, extend='max')
        ax.format(title=title, latlabels='l' if i == 0 else False,
                  lonlabels='b', **KWMAP)
        fig.colorbar(m, loc='b', col=i + 1, label=title)
    fig.format(suptitle='Orography')
    pplt.show()
else:
    print('No orography maps available')

In [ ]:
# Fig 4 — Does skill degrade with terrain complexity?
names = [n for n in ORDER if n in MODELPRED]
obs   = obsflat[valid]

orogpanels = []
if sdoraw is not None:
    orogpanels.append(('SDO (m)', sdoraw[valid]))
if seflat is not None:
    orogpanels.append(('Surface Elevation (m)', seflat[valid]))

if orogpanels:
    fig, axs = pplt.subplots(nrows=1, ncols=len(orogpanels),
                              refwidth=3, refheight=2, share=False)
    axsf = np.atleast_1d(axs).ravel()
    for ax, (oroglabel, orog) in zip(axsf, orogpanels):
        lo, hi = np.percentile(orog, [2, 98])
        edges  = np.linspace(lo, hi, 11)
        xc     = 0.5 * (edges[:-1] + edges[1:])
        idxs   = np.clip(np.digitize(orog, edges) - 1, 0, 9)
        for name in names:
            pred = MODELPRED[name][valid]
            r2s  = [r2(obs[idxs == i], pred[idxs == i])
                    if (idxs == i).sum() >= 500 else np.nan for i in range(10)]
            ax.plot(xc, r2s, color=COLORS[name], linewidth=1.5, marker='o',
                    markersize=3, label=LABELS[name])
        ax.format(xlabel=oroglabel, ylabel=r'$R^2$',
                  title=f'Skill vs. {oroglabel}', grid=False)
        ax.legend(loc='ur', ncols=1, frame=False)
    fig.format(suptitle=r'$R^2$ vs. Orography')
    pplt.show()
else:
    print('No orography data available')

In [ ]:
# Fig 5 — 1D PDP: mean precipitation vs each predictor
obs   = obsflat[valid]
names = [n for n in ORDER if n in MODELPRED]
plotvars = [v for v in ['rh', 'thetae', 'thetaestar', 'bl', 'lf', 'shf', 'lhf',
                        'sdo', 'sef', 'se'] if v in VARDATA]

binned = {v: bin_1d(VARDATA[v][valid], obs) for v in plotvars}
maxcnt = max(c.max() for _, _, c in binned.values())

ncols = min(4, len(plotvars))
nrows = -(-len(plotvars) // ncols)
fig, axs = pplt.subplots(nrows=nrows, ncols=ncols, refwidth=1.8, refheight=1.5,
                          sharex=False, sharey=True)
axsf = np.atleast_1d(axs).ravel()

for i, (ax, v) in enumerate(zip(axsf, plotvars)):
    xc, obsbin, counts = binned[v]
    dax = ax.twinx()
    dax.bar(xc, counts, width=xc[1] - xc[0],
            edgecolor='none', color='gray5', alpha=0.3)
    dax.format(ylim=(0, 1.5 * maxcnt), yformatter='sci')
    if i % ncols == ncols - 1 or i == len(plotvars) - 1:
        dax.format(ylabel='Count')
    else:
        dax.tick_params(axis='y', labelright=False)

    ax.plot(xc, obsbin, color='k', linewidth=2, zorder=5)
    for name in names:
        _, predbin, _ = bin_1d(VARDATA[v][valid], MODELPRED[name][valid])
        ax.plot(xc, predbin, color=COLORS[name], linewidth=1.5, zorder=5)
    ax.format(grid=False, xlabel=VARLABELS[v],
              ylabel='Precipitation (mm)' if i % ncols == 0 else '')

for ax in axsf[len(plotvars):]:
    ax.set_visible(False)

handles = [Line2D([], [], color='k', linewidth=2, label='Observed')]
handles += [Line2D([], [], color=COLORS[n], linewidth=1.5, label=LABELS[n])
            for n in names]
fig.legend(handles, loc='b', ncols=len(handles))
fig.format(suptitle='1D Partial Dependence')
pplt.show()

In [ ]:
# Fig 6 — 2D PDP: response surface for variable pairs
obs   = obsflat[valid]
names = [n for n in ORDER if n in MODELPRED]
pairs = [(px, py) for px, py in
         [('rh', 'thetae'), ('rh', 'thetaestar'), ('thetae', 'thetaestar')]
         if px in VARDATA and py in VARDATA]

if pairs and names:
    nrows_p = len(pairs)
    ncols_p = 1 + len(names)
    fig, axs = pplt.subplots(nrows=nrows_p, ncols=ncols_p,
                              refwidth=1.5, share=False)
    axsf = np.atleast_1d(axs).ravel()

    m_obs = m_bias = None
    for row, (px, py) in enumerate(pairs):
        x, y = VARDATA[px][valid], VARDATA[py][valid]
        xc, yc, obsbin, obsc = bin_2d(x, y, obs)

        ax = axsf[row * ncols_p]
        m_obs = ax.pcolormesh(xc, yc, obsbin.T, cmap='Blues',
                               vmin=0, vmax=3, extend='max')
        ax.contour(xc, yc, obsc.T, color='red3', linewidths=0.5)
        ax.format(title='Observed' if row == 0 else '',
                  ylabel=VARLABELS[py],
                  xlabel=VARLABELS[px] if row == nrows_p - 1 else '',
                  grid=False)

        for col, name in enumerate(names):
            ax = axsf[row * ncols_p + col + 1]
            _, _, predbin, _ = bin_2d(x, y, MODELPRED[name][valid])
            m_bias = ax.pcolormesh(xc, yc, (predbin - obsbin).T,
                                    cmap='DryWet', vmin=-2, vmax=2, extend='both')
            ax.contour(xc, yc, obsc.T, color='red3', linewidths=0.5)
            ax.format(title=LABELS.get(name, name) if row == 0 else '',
                      xlabel=VARLABELS[px] if row == nrows_p - 1 else '',
                      grid=False)

    if m_obs:
        fig.colorbar(m_obs, loc='l', label='Precipitation (mm)')
    if m_bias:
        fig.colorbar(m_bias, loc='r', label='Model − Observed (mm)')
    fig.format(suptitle='2D Partial Dependence')
    pplt.show()
else:
    print('Insufficient data for 2D PDP')

In [ ]:
# Fig 7 — Moisture vs Instability regime in M-I space
if 'sr_med' not in SR_REGISTRY:
    print('sr_med not in SR_REGISTRY — skipping M vs I plot')
else:
    c     = SR_REGISTRY['sr_med']['constants']
    obs   = obsflat[valid]
    names = [n for n in ORDER if n in MODELPRED]
    M = rhk[valid]
    I = thetaek[valid] - c['b'] * thetaestark[valid] - c['c']

    Mlo, Mhi = float(np.percentile(M, 2)), float(np.percentile(M, 98))
    Ilo, Ihi = float(np.percentile(I, 2)), float(np.percentile(I, 98))
    axlo, axhi = min(Mlo, Ilo), max(Mhi, Ihi)

    panels = [('Observed', obs)]
    for name in names:
        panels.append((LABELS.get(name, name), MODELPRED[name][valid]))

    fig, axs = pplt.subplots(nrows=1, ncols=len(panels), refwidth=2, share=True)
    axsf = np.atleast_1d(axs).ravel()

    for i, (ax, (title, z)) in enumerate(zip(axsf, panels)):
        xc, yc, zbin, _ = bin_2d(M, I, z, nbins=35, minsamples=20, plo=2, phi=98)
        m = ax.pcolormesh(xc, yc, zbin.T, cmap='ColdHot_r',
                          cmap_kw={'left': 0.5}, vmin=0, vmax=4,
                          levels=18, extend='max')
        ax.plot([axlo, axhi], [axlo, axhi], 'k--', lw=1)
        ax.format(title=title,
                  xlabel=r'$M$ (moisture)',
                  ylabel=r'$I$ (instability)' if i == 0 else '',
                  grid=False, xlim=(Mlo, Mhi), ylim=(Ilo, Ihi))

    fig.colorbar(m, loc='r', label='Precipitation (mm)')
    fig.format(suptitle='Moisture vs. Instability Regime')
    pplt.show()

In [ ]:
# Fig 8 — Precipitation distribution and conditional bias
names = [n for n in ORDER if n in MODELPRED]
obs   = obsflat[valid]
hi    = np.percentile(obs[obs > 0], 99.9)
edges = np.linspace(0, hi, 60)
xc    = 0.5 * (edges[:-1] + edges[1:])

fig, axs = pplt.subplots(nrows=1, ncols=2, refwidth=3, refheight=2, share=False)

obshist, _ = np.histogram(obs, bins=edges, density=True)
axs[0].plot(xc, obshist, color='k', linewidth=2, label='Observed', zorder=6)
for name in names:
    predhist, _ = np.histogram(MODELPRED[name][valid], bins=edges, density=True)
    axs[0].plot(xc, predhist, color=COLORS[name], linewidth=1.5, label=LABELS[name])
axs[0].format(grid=False, xlabel='Total Precipitation (mm)', ylabel='Density',
              title='PDF', yscale='log')
axs[0].legend(loc='ur', ncols=1, frame=False)

xc_b, _, _ = bin_1d(obs, obs, nbins=20, minsamples=100, plo=0, phi=99)
axs[1].axhline(0, color='gray', linewidth=0.8, linestyle='--')
for name in names:
    _, biasbin, _ = bin_1d(obs, MODELPRED[name][valid] - obs,
                           nbins=20, minsamples=100, plo=0, phi=99)
    axs[1].plot(xc_b, biasbin, color=COLORS[name], linewidth=1.5, label=LABELS[name])
axs[1].format(grid=False, xlabel='Observed Precipitation (mm)',
              ylabel='Predicted − Observed (mm)', title='Conditional Bias')
axs[1].legend(loc='ll', ncols=1, frame=False)

fig.format(suptitle='Precipitation Distribution and Bias')
pplt.show()

In [ ]:
# Fig 9 — Feature correlations with model residuals + binned log-residual curves
names = [n for n in ORDER if n in MODELPRED]
obs   = obsflat[valid]

def feat_corrs(residual):
    out = {}
    for vname, arr in VARDATA.items():
        x = arr[valid]
        m = np.isfinite(x) & np.isfinite(residual)
        if m.sum() > 100:
            out[vname] = np.corrcoef(x[m], residual[m])[0, 1]
    return out

all_corrs = {name: feat_corrs(obs - MODELPRED[name][valid]) for name in names}
all_feats = sorted(set().union(*[c.keys() for c in all_corrs.values()]))
max_abs   = {f: max(abs(all_corrs.get(n, {}).get(f, 0)) for n in names)
             for f in all_feats}
ranked    = sorted(all_feats, key=lambda f: max_abs[f], reverse=True)

x     = np.arange(len(ranked))
width = 0.8 / max(len(names), 1)
fig, ax = pplt.subplots(refwidth=4, refheight=3)
for i, name in enumerate(names):
    vals = [all_corrs.get(name, {}).get(f, 0) for f in ranked]
    ax.barh(x + i * width, vals, height=width,
            color=COLORS[name], alpha=0.85 - 0.15 * i, label=LABELS[name])
ax.format(yticks=x + width * (len(names) - 1) / 2,
          yticklabels=[VARLABELS.get(f, f) for f in ranked],
          xlabel='Pearson r (feature vs. residual)',
          title='Feature–Residual Correlation', grid=False)
ax.axvline(0, color='gray', linewidth=0.7)
ax.legend(loc='lr', ncols=1, frame=False)
pplt.show()

plotfeats = [f for f in ranked if f in VARDATA]
ncols_r   = min(4, len(plotfeats))
nrows_r   = -(-len(plotfeats) // ncols_r)
fig, axs  = pplt.subplots(nrows=nrows_r, ncols=ncols_r, refwidth=1.8,
                           refheight=1.5, sharex=False, sharey=True)
axsf = np.atleast_1d(axs).ravel()
for i, (ax, f) in enumerate(zip(axsf, plotfeats)):
    ax.axhline(0, color='gray', linewidth=0.7, linestyle='--')
    for name in names:
        resid_log = np.log1p(np.maximum(0, obs)) - \
                    np.log1p(np.maximum(0, MODELPRED[name][valid]))
        xc, means, _ = bin_1d(VARDATA[f][valid], resid_log)
        ax.plot(xc, means, color=COLORS[name], linewidth=1.5)
    ax.format(grid=False, xlabel=VARLABELS.get(f, f),
              ylabel='log-residual' if i % ncols_r == 0 else '')
for ax in axsf[len(plotfeats):]:
    ax.set_visible(False)
handles = [Line2D([], [], color=COLORS[n], linewidth=1.5, label=LABELS[n])
           for n in names]
fig.legend(handles, loc='b', ncols=len(names))
fig.format(suptitle='log(1+obs) − log(1+pred) vs. Features')
pplt.show()

In [ ]:
# Fig 10 — Distillation gap: which features explain the NN-GAUSS vs SR-HI difference?
has_gap = 'nn_gauss' in MODELPRED and 'sr_hi' in MODELPRED
if has_gap:
    obs     = obsflat[valid]
    nn_pred = MODELPRED['nn_gauss'][valid]
    sr_pred = MODELPRED['sr_hi'][valid]

    gap_add   = nn_pred - sr_pred
    gap_log   = np.log1p(nn_pred) - np.log1p(sr_pred)
    truth_gap = obs - sr_pred

    gaps = {'NN − SR (add)': gap_add,
            'NN − SR (log)': gap_log,
            'Obs − SR (add)': truth_gap}

    results = {}
    for gname, gap in gaps.items():
        corrs = {}
        for vname, arr in VARDATA.items():
            x = arr[valid]
            m = np.isfinite(x) & np.isfinite(gap)
            if m.sum() > 100:
                corrs[vname] = np.corrcoef(x[m], gap[m])[0, 1]
        results[gname] = corrs

    ref_corrs = results['NN − SR (add)']
    ranked = sorted(ref_corrs.keys(), key=lambda f: abs(ref_corrs[f]), reverse=True)

    print('Feature correlations with distillation gap (NN-GAUSS − SR-HI):')
    for f in ranked:
        line = f'  {VARLABELS.get(f, f):30s}'
        for gn in gaps:
            r = results[gn].get(f, 0)
            line += f'  {gn}: r={r:+.3f}'
        print(line)

    gap_names  = list(gaps.keys())
    gap_colors = ['#2355a1', '#539ed4', '#D42028']
    x     = np.arange(len(ranked))
    width = 0.8 / len(gap_names)

    fig, ax = pplt.subplots(refwidth=4, refheight=3)
    for gi, gn in enumerate(gap_names):
        vals = [results[gn].get(f, 0) for f in ranked]
        ax.barh(x + gi * width, vals, height=width,
                color=gap_colors[gi], label=gn)
    ax.format(yticks=x + width,
              yticklabels=[VARLABELS.get(f, f) for f in ranked],
              xlabel='Pearson r',
              title='Feature Correlation with Distillation Gap', grid=False)
    ax.axvline(0, color='gray', linewidth=0.7)
    ax.legend(loc='lr', ncols=1, frame=False)
    pplt.show()

    top_feats = ranked[:6]
    ncols_g   = min(3, len(top_feats))
    nrows_g   = -(-len(top_feats) // ncols_g)
    fig, axs  = pplt.subplots(nrows=nrows_g, ncols=ncols_g, refwidth=1.8,
                               refheight=1.5, sharex=False, sharey=True)
    axsf = np.atleast_1d(axs).ravel()
    for i, (ax, f) in enumerate(zip(axsf, top_feats)):
        ax.axhline(0, color='gray', linewidth=0.7, linestyle='--')
        xc, means, _ = bin_1d(VARDATA[f][valid], gap_add)
        ax.plot(xc, means, color='#2355a1', linewidth=1.5)
        xc_l, means_l, _ = bin_1d(VARDATA[f][valid], gap_log)
        ax.plot(xc_l, means_l, color='#539ed4', linewidth=1.5, linestyle='--')
        ax.format(grid=False, xlabel=VARLABELS.get(f, f),
                  ylabel='Gap' if i % ncols_g == 0 else '')
    for ax in axsf[len(top_feats):]:
        ax.set_visible(False)
    handles = [Line2D([], [], color='#2355a1', linewidth=1.5, label='Additive'),
               Line2D([], [], color='#539ed4', linewidth=1.5, linestyle='--',
                      label='Log-space')]
    fig.legend(handles, loc='b', ncols=2)
    fig.format(suptitle='Distillation Gap vs. Features')
    pplt.show()
else:
    print('Need both nn_gauss and sr_hi for distillation gap analysis')

In [ ]:
# Fig 11 — Pareto fronts from SR search runs
PARETO_RUNS = ['sr_gauss', 'sr_gauss_v2', 'sr_gauss_v3']

pareto_colors  = ['#2355a1', '#539ed4', '#89c4e8', '#D42028', '#F2C85E']
pareto_markers = ['o', 's', 'D', '^', 'v']
pareto_labels  = {
    'sr_gauss':    'sr_gauss (LF, SHF, LHF)',
    'sr_gauss_v2': 'sr_gauss_v2 (LF, SHF, LHF, SEF, BOW, SE, SDO)',
    'sr_gauss_v3': 'sr_gauss_v3 (SE, SDO)'}

paretos = {}
for run in PARETO_RUNS:
    p = load_pareto(run)
    if p is not None:
        paretos[run] = p

if paretos:
    fig, ax = pplt.subplots(refwidth=4, refheight=2.5)

    for i, (run, p) in enumerate(paretos.items()):
        ax.plot(p['complexity'], p['loss'],
                color=pareto_colors[i % len(pareto_colors)],
                linewidth=1.5,
                marker=pareto_markers[i % len(pareto_markers)],
                markersize=3,
                label=pareto_labels.get(run, run), zorder=3)

    opt_path = os.path.join(MODELSDIR, 'sr', 'optimized_equations.csv')
    if os.path.exists(opt_path):
        opt = pd.read_csv(opt_path)
        for refname in ['sr_hi', 'sr_med']:
            refval = opt.loc[opt['name'] == refname, 'valid_loss'].values
            if len(refval):
                ax.axhline(refval[0], color='gray', linewidth=0.7,
                           linestyle='--', zorder=1)
                ax.text(2, refval[0] + 0.005,
                        f'{LABELS.get(refname, refname)} ({refval[0]:.3f})',
                        fontsize=7, color='gray')

    ax.format(grid=False, xlabel='Equation Complexity',
              ylabel='Loss (mean over seeds)',
              title='Pareto Fronts', xlim=(0, 31))
    ax.legend(loc='ur', ncols=1, frame=False)
    pplt.show()
else:
    print('No Pareto CSV files found')